# 20 — Novo CAGED

Baixa microdados mensais do Novo CAGED para os municípios selecionados. O notebook descobre as competências disponíveis e mantém o Novo CAGED separado de séries históricas anteriores a 2020.


In [ ]:
%pip -q install google-cloud-bigquery pandas pyarrow db-dtypes tqdm


In [ ]:
from pathlib import Path
import os, json, math
import pandas as pd

ROOT = Path.cwd()
DATA_DIR = ROOT / "dados"
OUT_DIR = DATA_DIR / "processado"
CONTROL_DIR = DATA_DIR / "controle"
for p in [DATA_DIR, OUT_DIR, CONTROL_DIR]: p.mkdir(parents=True, exist_ok=True)

ARQUIVO_MUNICIPIOS = ROOT / "municipios.csv"
MUNICIPIOS_INLINE = [
    # "3516408",  # Franco da Rocha
    # "3525904",  # Jundiaí
]
LOTE_TAMANHO = 5

if ARQUIVO_MUNICIPIOS.exists():
    mun = pd.read_csv(ARQUIVO_MUNICIPIOS, dtype=str)
    if "id_municipio" not in mun.columns:
        raise ValueError("municipios.csv deve conter a coluna id_municipio")
    ids = mun["id_municipio"].astype(str).str.strip().dropna().tolist()
else:
    ids = [str(x).strip() for x in MUNICIPIOS_INLINE if str(x).strip()]

ids = list(dict.fromkeys(ids))
if not ids:
    raise ValueError("Informe municípios em municipios.csv ou em MUNICIPIOS_INLINE.")
if any(len(x) != 7 or not x.isdigit() for x in ids):
    raise ValueError("Todos os códigos devem ser códigos IBGE municipais de 7 dígitos.")

LOTES = [ids[i:i+LOTE_TAMANHO] for i in range(0, len(ids), LOTE_TAMANHO)]
print(f"Municípios: {len(ids)} | lotes: {len(LOTES)} | tamanho máximo: {LOTE_TAMANHO}")

from google.cloud import bigquery
BILLING_PROJECT_ID=os.getenv("BIGQUERY_PROJECT")
if not BILLING_PROJECT_ID:
    raise EnvironmentError("Defina BIGQUERY_PROJECT com um projeto Google Cloud habilitado para cobrança do BigQuery.")
bq=bigquery.Client(project=BILLING_PROJECT_ID)
TABLE="basedosdados.br_me_caged.microdados_movimentacao"


In [ ]:
PERIODOS=[(int(r.ano),int(r.mes)) for r in bq.query(f"SELECT DISTINCT ano,mes FROM `{TABLE}` WHERE ano IS NOT NULL AND mes IS NOT NULL ORDER BY ano,mes").result()]
print("Cobertura:", PERIODOS[0], "a", PERIODOS[-1], "| competências:", len(PERIODOS))


In [ ]:
from tqdm.auto import tqdm
base=OUT_DIR/"caged"; base.mkdir(parents=True,exist_ok=True)
for lote_n,lote in enumerate(LOTES,1):
    ids_sql=','.join(f"'{x}'" for x in lote)
    for ano,mes in tqdm(PERIODOS, desc=f"Novo CAGED lote {lote_n:02d}"):
        out=base/f"novo_caged_lote{lote_n:02d}_{ano}_{mes:02d}.parquet"
        if out.exists(): continue
        q=f"SELECT * FROM `{TABLE}` WHERE id_municipio IN ({ids_sql}) AND ano={ano} AND mes={mes}"
        df=bq.query(q).to_dataframe(create_bqstorage_client=False)
        df["regime_caged"]="novo_caged"
        df.to_parquet(out,index=False,compression="snappy")
print("Novo CAGED concluído em", base)
